In [ ]:
# INSTALL REQUIRED LIBRARIES

!pip -q install xgboost

In [ ]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.svm import SVR

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

In [ ]:
# UPLOAD DATASET

uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)

print("Dataset loaded successfully!")
print("File:", filename)
print("Shape:", df.shape)

In [ ]:
# INITIAL DATA INSPECTION

print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

print("\nDataset Information:")
df.info()

In [ ]:
# STANDARDISE COLUMN NAMES

df.columns = df.columns.str.strip()

print(df.columns.tolist())

In [ ]:
# CHECK DUPLICATES

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

In [ ]:
# REMOVE DUPLICATES

df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

In [ ]:
# HANDLE INVALID VALUES

df = df.replace(
    ["-", "N/A", "NA", "na", "n/a", "", "null", "None"],
    np.nan
)

print("Invalid values converted to NaN.")

In [ ]:
# CONVERT NUMERICAL COLUMNS

numeric_columns = [
    "AQI",
    "PM2.5",
    "PM10",
    "O3",
    "NO2",
    "SO2",
    "CO",
    "Latitude",
    "Longitude"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_columns].dtypes)

In [ ]:
# CONVERT TIME COLUMN

df["Time"] = pd.to_datetime(
    df["Time"],
    errors="coerce"
)

print(df["Time"].dtype)

print("\nTime range:")
print(df["Time"].min())
print(df["Time"].max())

In [ ]:
# MISSING VALUE ANALYSIS

missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) * 100
})

missing = missing.sort_values(
    "Missing Percentage",
    ascending=False
)

display(missing)

In [ ]:
# MISSING VALUE HANDLING

# Check missing values before handling
print("Missing values before handling:")
display(df.isnull().sum())

# Handle missing Time values

df = df.dropna(subset=["Time"]).copy()

# Handle missing AQI values

df = df.dropna(subset=["AQI"]).copy()

# Keep missing pollutant values for now

print("\nMissing values after removing missing Time and AQI:")
display(df.isnull().sum())

print("\nDataset shape after handling target/time missing values:")
print(df.shape)

In [ ]:
# EDA
# AQI BASIC STATISTICS

print(df["AQI"].describe())

In [ ]:
# AQI DISTRIBUTION

plt.figure(figsize=(6, 4))

sns.histplot(
    df["AQI"],
    bins=40,
    kde=True
)

plt.title("Distribution of Air Quality Index (AQI)")
plt.xlabel("AQI")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# CORRELATION ANALYSIS

corr_columns = [
    "AQI",
    "PM2.5",
    "PM10",
    "O3",
    "NO2",
    "SO2",
    "CO",
    "Latitude",
    "Longitude"
]

correlation = df[corr_columns].corr()

plt.figure(figsize=(6, 4))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix of Air Quality Variables")
plt.show()

In [ ]:
# AQI BY MAJOR STATIONS

top_cities = df["City"].value_counts().head(15).index

city_data = df[df["City"].isin(top_cities)]

plt.figure(figsize=(8, 4))

sns.boxplot(
    data=city_data,
    x="AQI",
    y="City"
)

plt.title("AQI Distribution for Major Stations")
plt.xlabel("AQI")
plt.ylabel("Station / City")

plt.tight_layout()
plt.show()

In [ ]:
# TIME-BASED FEATURE ENGINEERING

df["Year"] = df["Time"].dt.year
df["Month"] = df["Time"].dt.month
df["Day"] = df["Time"].dt.day
df["Hour"] = df["Time"].dt.hour
df["DayOfWeek"] = df["Time"].dt.dayofweek
df["DayOfYear"] = df["Time"].dt.dayofyear

print(df[
    [
        "Time",
        "Year",
        "Month",
        "Day",
        "Hour",
        "DayOfWeek",
        "DayOfYear"
    ]
].head())

In [ ]:
# AQI BY HOUR

hourly_aqi = df.groupby("Hour")["AQI"].mean()

plt.figure(figsize=(8, 4))

hourly_aqi.plot(
    marker="o"
)

plt.title("Average AQI by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Average AQI")

plt.xticks(range(0, 24))

plt.show()

In [ ]:
# AQI BY MONTH

monthly_aqi = df.groupby("Month")["AQI"].mean()

plt.figure(figsize=(8, 4))

monthly_aqi.plot(
    marker="o"
)

plt.title("Average AQI by Month")
plt.xlabel("Month")
plt.ylabel("Average AQI")

plt.xticks(range(1, 13))

plt.show()

In [ ]:
# AQI OVER TIME

daily_aqi = (
    df.groupby(df["Time"].dt.date)["AQI"]
    .mean()
)

plt.figure(figsize=(8, 4))

plt.plot(
    daily_aqi.index,
    daily_aqi.values
)

plt.title("Average AQI Over Time")
plt.xlabel("Date")
plt.ylabel("Average AQI")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# DEFINE FEATURES AND TARGET

X = df_model[
    [
        "City",
        "PM2.5",
        "PM10",
        "O3",
        "NO2",
        "SO2",
        "CO",
        "Latitude",
        "Longitude",
        "Year",
        "Month",
        "Day",
        "Hour",
        "DayOfWeek",
        "DayOfYear"
    ]
]

y = df_model["AQI"]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# CHRONOLOGICAL TRAIN-TEST SPLIT

df_model = df_model.sort_values("Time").reset_index(drop=True)

X = df_model[
    [
        "City",
        "PM2.5",
        "PM10",
        "O3",
        "NO2",
        "SO2",
        "CO",
        "Latitude",
        "Longitude",
        "Year",
        "Month",
        "Day",
        "Hour",
        "DayOfWeek",
        "DayOfYear"
    ]
]

y = df_model["AQI"]

split_index = int(len(df_model) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

print("\nTraining period:")
print(df_model["Time"].iloc[:split_index].min())
print("to")
print(df_model["Time"].iloc[:split_index].max())

print("\nTesting period:")
print(df_model["Time"].iloc[split_index:].min())
print("to")
print(df_model["Time"].iloc[split_index:].max())

In [ ]:
# PREPROCESSING

categorical_features = [
    "City"
]

numerical_features = [
    "PM2.5",
    "PM10",
    "O3",
    "NO2",
    "SO2",
    "CO",
    "Latitude",
    "Longitude",
    "Year",
    "Month",
    "Day",
    "Hour",
    "DayOfWeek",
    "DayOfYear"
]

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [ ]:
# DEFINE MACHINE LEARNING MODELS

models = {

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ),

    "SVR": SVR(
        kernel="rbf",
        C=100,
        epsilon=0.1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

In [ ]:
# TRAIN THE MODELS

trained_models = {}

predictions = {}

for name, model in models.items():

    print("=" * 60)
    print("Training:", name)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(
        X_train,
        y_train
    )

    y_pred = pipeline.predict(X_test)

    trained_models[name] = pipeline
    predictions[name] = y_pred

    print("Training completed.")

In [ ]:
# MAPE FUNCTION

def calculate_mape(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mask = y_true != 0

    return np.mean(
        np.abs(
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100

In [ ]:
# MODEL EVALUATION

results = []

for name, y_pred in predictions.items():

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred
        )
    )

    r2 = r2_score(
        y_test,
        y_pred
    )

    mape = calculate_mape(
        y_test,
        y_pred
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE (%)": mape
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "RMSE"
).reset_index(drop=True)

display(results_df)

In [ ]:
# MODEL COMPARISON

metrics = [
    "MAE",
    "RMSE",
    "MAPE (%)"
]

for metric in metrics:

    plt.figure(figsize=(6, 4))

    sns.barplot(
        data=results_df,
        x="Model",
        y=metric
    )

    plt.title(f"Model Comparison - {metric}")
    plt.xticks(rotation=20)

    plt.tight_layout()
    plt.show()

In [ ]:
# R-SQUARED COMPARISON

plt.figure(figsize=(6, 4))

sns.barplot(
    data=results_df,
    x="Model",
    y="R2"
)

plt.title("Model Comparison - R²")
plt.ylabel("R² Score")
plt.xticks(rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# XGBOOST FEATURE IMPORTANCE

xgb_pipeline = trained_models["XGBoost"]

xgb_preprocessor = xgb_pipeline.named_steps["preprocessor"]

xgb_model = xgb_pipeline.named_steps["model"]

xgb_features = xgb_preprocessor.get_feature_names_out()

xgb_importance = xgb_model.feature_importances_

xgb_importance_df = pd.DataFrame({
    "Feature": xgb_features,
    "Importance": xgb_importance
})

xgb_importance_df = xgb_importance_df.sort_values(
    "Importance",
    ascending=False
)

In [ ]:
# XGBOOST FEATURE IMPORTANCE PLOT

top_xgb = xgb_importance_df.head(15)

plt.figure(figsize=(8, 4))

sns.barplot(
    data=top_xgb,
    x="Importance",
    y="Feature"
)

plt.title(
    "Top 15 Features - XGBoost"
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

In [ ]:
# INTERACTIVE AQI PREDICTION SYSTEM

import ipywidgets as widgets
from IPython.display import display, clear_output

# Input widgets
city = widgets.Dropdown(
    options=sorted(df["City"].dropna().unique()),
    value=sorted(df["City"].dropna().unique())[0],
    description="City:"
)

pm25 = widgets.FloatSlider(
    min=0, max=300, step=0.1, value=88, description="PM2.5:"
)

pm10 = widgets.FloatSlider(
    min=0, max=300, step=0.1, value=71, description="PM10:"
)

o3 = widgets.FloatSlider(
    min=0, max=300, step=0.1, value=26.3, description="O3:"
)

no2 = widgets.FloatSlider(
    min=0, max=300, step=0.1, value=4.6, description="NO2:"
)

so2 = widgets.FloatSlider(
    min=0, max=300, step=0.1, value=2.4, description="SO2:"
)

co = widgets.FloatSlider(
    min=0, max=20, step=0.1, value=6.5, description="CO:"
)

latitude = widgets.FloatText(
    value=14.67589, description="Latitude:"
)

longitude = widgets.FloatText(
    value=77.59303, description="Longitude:"
)

year = widgets.IntText(
    value=2024, description="Year:"
)

month = widgets.IntSlider(
    min=1, max=12, step=1, value=5, description="Month:"
)

day = widgets.IntSlider(
    min=1, max=31, step=1, value=4, description="Day:"
)

hour = widgets.IntSlider(
    min=0, max=23, step=1, value=18, description="Hour:"
)

# Prediction button
predict_btn = widgets.Button(
    description="Generate AQI Prediction",
    button_style="primary"
)

output = widgets.Output()


# Prediction function
def on_button_clicked(b):

    with output:
        clear_output()

        prediction = predict_aqi(
            city.value,
            pm25.value,
            pm10.value,
            o3.value,
            no2.value,
            so2.value,
            co.value,
            latitude.value,
            longitude.value,
            year.value,
            month.value,
            day.value,
            hour.value
        )

        print("=" * 40)
        print("       AIR QUALITY PREDICTION")
        print("=" * 40)
        print(f"Model Used     : XGBoost")
        print(f"Predicted AQI  : {prediction:.2f}")
        print("=" * 40)


predict_btn.on_click(on_button_clicked)


# Dashboard layout
dashboard = widgets.VBox([
    widgets.HTML("<h2>🌍 Air Quality Prediction System</h2>"),

    widgets.HBox([
        widgets.VBox([city, pm25, pm10, o3, no2, so2]),
        widgets.VBox([co, latitude, longitude, year, month, day, hour])
    ]),

    predict_btn,
    output
])

display(dashboard)